# Assignment 6: Object Detection using Transfer Learning (VGG16 on CIFAR-10)
a. Load a pre-trained CNN (VGG16, trained on ImageNet)
b. Freeze the lower convolutional layers
c. Add a custom classifier with trainable layers
d. Train the classifier layers
e. Fine-tune: unfreeze more layers and retrain with a smaller learning rate

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from tensorflow.keras.applications import VGG16
import os

I0000 00:00:1793640651.777803   37599 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1793640653.554382   37599 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Load data (subset of CIFAR-10 so training is fast on CPU)

In [2]:
# Path of dataset folder (change if your folder is somewhere else)
train_dir = "dataset/cifar-10-img/train"
test_dir  = "dataset/cifar-10-img/test"

datagen = ImageDataGenerator(rescale=1.0/255)   # pixel values 0-255 -> 0-1

train_gen = datagen.flow_from_directory(train_dir, target_size=(32, 32),
                                        color_mode="rgb", batch_size=5000,
                                        class_mode="categorical", shuffle=True)
test_gen  = datagen.flow_from_directory(test_dir, target_size=(32, 32),
                                        color_mode="rgb", batch_size=1000,
                                        class_mode="categorical", shuffle=True)

# One big batch = whole dataset loaded as numpy arrays
x_train, y_train = train_gen[0]
x_test,  y_test  = test_gen[0]
class_names = list(train_gen.class_indices.keys())

print("x_train:", x_train.shape, " y_train:", y_train.shape)
print("x_test :", x_test.shape,  " y_test :", y_test.shape)
print("Classes:", class_names)

Found 40079 images belonging to 10 classes.


Found 9921 images belonging to 10 classes.


x_train: (5000, 32, 32, 3)  y_train: (5000, 10)
x_test : (1000, 32, 32, 3)  y_test : (1000, 10)
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [3]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[np.argmax(y_train[i])])
    plt.axis("off")
plt.show()

/tmp/ipykernel_37599/676615008.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## a. Load pre-trained VGG16 (without its top classifier)

In [4]:
# Offline: put "vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5" next to this notebook.
# Online: weights are downloaded automatically from the internet.
weights_path = "vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
weights = weights_path if os.path.exists(weights_path) else "imagenet"

base_model = VGG16(weights=weights, include_top=False, input_shape=(32, 32, 3))
base_model.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

## b. Freeze parameters (weights) of the convolutional layers

In [5]:
for layer in base_model.layers:
    layer.trainable = False

## c. Add custom classifier

In [6]:
x = layers.Flatten()(base_model.output)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs=base_model.input, outputs=outputs)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 32, 32, 64)     │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 32, 32, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 16, 16, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 8, 8, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 8, 8, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 4, 4, 512)      │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 4, 4, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 2, 2, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 2, 2, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 1, 1, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         2,57

 Total params: 14,914,378 (56.89 MB)

 Trainable params: 199,690 (780.04 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

## d. Train classifier layers

In [7]:
history = model.fit(x_train, y_train, batch_size=64, epochs=10,
                    validation_data=(x_test, y_test))

Epoch 1/10


 1/79 ━━━━━━━━━━━━━━━━━━━━ 1:51 1s/step - accuracy: 0.1094 - loss: 2.3822

 2/79 ━━━━━━━━━━━━━━━━━━━━ 8s 109ms/step - accuracy: 0.0938 - loss: 2.3971

 3/79 ━━━━━━━━━━━━━━━━━━━━ 8s 110ms/step - accuracy: 0.0938 - loss: 2.3743

 4/79 ━━━━━━━━━━━━━━━━━━━━ 8s 112ms/step - accuracy: 0.1016 - loss: 2.3514

 5/79 ━━━━━━━━━━━━━━━━━━━━ 8s 114ms/step - accuracy: 0.1063 - loss: 2.3266

... (791 outputs left out)

76/79 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.6486 - loss: 1.0010

77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.6496 - loss: 0.9990

78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.6486 - loss: 1.0005

79/79 ━━━━━━━━━━━━━━━━━━━━ 8s 95ms/step - accuracy: 0.6492 - loss: 0.9994 - val_accuracy: 0.5200 - val_loss: 1.3814


In [8]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.show()

/tmp/ipykernel_37599/1834603888.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## e. Fine-tune: unfreeze the last 4 layers of VGG16 and use a small learning rate

In [9]:
for layer in base_model.layers[-4:]:
    layer.trainable = True

model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-5),
              loss="categorical_crossentropy", metrics=["accuracy"])

history = model.fit(x_train, y_train, batch_size=64, epochs=10,
                    validation_data=(x_test, y_test))

Epoch 1/10


 1/79 ━━━━━━━━━━━━━━━━━━━━ 1:49 1s/step - accuracy: 0.6250 - loss: 1.0324

 2/79 ━━━━━━━━━━━━━━━━━━━━ 38s 500ms/step - accuracy: 0.6562 - loss: 0.9871

 3/79 ━━━━━━━━━━━━━━━━━━━━ 38s 503ms/step - accuracy: 0.6510 - loss: 0.9960

 4/79 ━━━━━━━━━━━━━━━━━━━━ 37s 499ms/step - accuracy: 0.6562 - loss: 0.9784

 5/79 ━━━━━━━━━━━━━━━━━━━━ 36s 498ms/step - accuracy: 0.6562 - loss: 0.9738

... (800 outputs left out)

77/79 ━━━━━━━━━━━━━━━━━━━━ 1s 513ms/step - accuracy: 0.8754 - loss: 0.3957

78/79 ━━━━━━━━━━━━━━━━━━━━ 0s 513ms/step - accuracy: 0.8758 - loss: 0.3956

79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 508ms/step - accuracy: 0.8760 - loss: 0.3950

79/79 ━━━━━━━━━━━━━━━━━━━━ 41s 525ms/step - accuracy: 0.8760 - loss: 0.3950 - val_accuracy: 0.5890 - val_loss: 1.3310


In [10]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="Train loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.title("Loss"); plt.xlabel("Epoch"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="Train accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.title("Accuracy"); plt.xlabel("Epoch"); plt.legend()
plt.show()

/tmp/ipykernel_37599/1834603888.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Evaluate and predict

In [11]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc*100:.2f}%")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=class_names))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt="d",
            xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion Matrix")
plt.show()

Test loss: 1.3310
Test accuracy: 58.90%


              precision    recall  f1-score   support

    airplane       0.67      0.58      0.62       108
  automobile       0.71      0.68      0.69        99
        bird       0.62      0.41      0.49       100
         cat       0.35      0.50      0.42        88
        deer       0.59      0.52      0.55       102
         dog       0.54      0.45      0.49       110
        frog       0.60      0.69      0.64        93
       horse       0.62      0.71      0.66        96
        ship       0.61      0.77      0.68       100
       truck       0.64      0.61      0.62       104

    accuracy                           0.59      1000
   macro avg       0.60      0.59      0.59      1000
weighted avg       0.60      0.59      0.59      1000



/tmp/ipykernel_37599/4274663811.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
pred = model.predict(x_test[:10], verbose=0)
plt.figure(figsize=(12, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_test[i]); plt.axis("off")
    plt.title(f"P: {class_names[np.argmax(pred[i])]}\nA: {class_names[np.argmax(y_test[i])]}")
plt.suptitle("Object Detection using Transfer Learning (P = predicted, A = actual)")
plt.show()

/tmp/ipykernel_37599/856853200.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
